# モデル学習の並列化ベンチマーク

前処理済みshardと同じ初期モデルを使い、全16エージェントのself/相手モデル32ジョブを並列実行して比較します。ゲーム側の`worker-batched`と同様に、論理CPU数−1の軽量workerがshard読込・shuffle・minibatch組立を担当し、PyTorch/MPSは中央の1プロセスだけが所有します。中央では異なるモデルのforwardを1 waveへまとめてMPSへ先行投入します。本番の`agents/16model_result`や`model_episodeXXXX`は更新せず、検証モデルもファイルへ保存しません。metrics・ログ・集計結果だけを`results/training_parallel_benchmark`へ残します。

既定値は全体の並列性能を検証する全32モデル×1エポックです。モデルは保存しません。本番の3エポックと計算量を揃える場合だけ`EPOCHS=3`に変更します。既定では最適化済みの最大worker構成を1回測り、1 workerとの比較も行う場合は`TRAINING_WORKERS=[1, CPU_WORKERS]`、再現性も測る場合は`REPEATS=2`にします。

In [10]:
from __future__ import annotations

import json
import os
from pathlib import Path
import re
import subprocess
import sys
import time

import pandas as pd
import psutil
from IPython.display import display


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'tools' / 'benchmark_parallel_training.py').exists():
            return candidate
    raise FileNotFoundError('pokemon-tcg-agent のリポジトリルートが見つかりません。')


def find_latest_complete_shards(match_root: Path) -> Path:
    candidates = []
    for path in (match_root / 'results' / 'training_loop').glob(
        '*/episode*/learning/shards'
    ):
        if len(list(path.glob('*/manifest.json'))) == 32:
            candidates.append(path)
    if not candidates:
        raise FileNotFoundError('32学習ジョブ分が揃った前処理shardがありません。')
    return max(candidates, key=lambda path: path.stat().st_mtime)


MATCH_ROOT = find_repo_root()
TRAIN_ROOT = MATCH_ROOT
venv_python = MATCH_ROOT / '.venv' / (
    'Scripts/python.exe' if os.name == 'nt' else 'bin/python'
)
PYTHON = venv_python if venv_python.exists() else Path(sys.executable)
BENCHMARK_RUNNER = MATCH_ROOT / 'tools' / 'benchmark_parallel_training.py'
TRAIN_SCRIPT = TRAIN_ROOT / 'tools' / 'train' / 'train_imitation.py'
AGENTS_ROOT = MATCH_ROOT / 'agents' / '16model_result'
SHARDS_ROOT = find_latest_complete_shards(MATCH_ROOT)
for required in (BENCHMARK_RUNNER, TRAIN_SCRIPT):
    if not required.exists():
        raise FileNotFoundError(required)

print(f'MATCH_ROOT={MATCH_ROOT}')
print(f'SHARDS_ROOT={SHARDS_ROOT}')
print(f'PYTHON={PYTHON}')

MATCH_ROOT=/Users/naoki/Desktop/develop/pokemon_tgc_agent/pokemon-tcg-agent
SHARDS_ROOT=/Users/naoki/Desktop/develop/pokemon_tgc_agent/pokemon-tcg-agent/results/training_loop/20260801_041621_030233/episode3600/learning/shards
PYTHON=/Users/naoki/Desktop/develop/pokemon_tgc_agent/pokemon-tcg-agent/.venv/bin/python


## 計測条件

`TRAINING_WORKERS`はshard前処理を行うCPU worker数で、既定ではゲーム並列化と同じく論理CPU数−1です。全構成で全32ジョブ、同じ初期パラメータ、seedを使用します。`SCHEDULE_ORDER='largest-first'`ではmanifestのサンプル数が多いジョブから割り当て、`DEVICE_WAVE_SIZE`件の異なるモデルbatchを中央MPSへ先行投入してからbackwardします。`samples/s`は端数を除いて実際にoptimizerへ渡した全モデル合計サンプル数÷wall秒、`games/s`は重複なしの元ゲーム数（現在3,600）÷wall秒です。ゲームを32モデル分に重複加算しません。

In [11]:
CPU_THREADS = os.cpu_count() or 1
CPU_WORKERS = max(1, CPU_THREADS - 1)
TRAINING_WORKERS = [CPU_WORKERS]
REPEATS = 1
JOB_LIMIT = 0  # self/相手モデルを合わせた全32ジョブ
EPOCHS = 1  # 並列性能の検証用。本番計算量で測る場合は3
TRAIN_MINIBATCH_SIZE = 128
LEARNING_RATE = 3e-4
DEVICE = 'auto'
SEED = 0
CENTRAL_DEVICE_OWNER = True
DEVICE_WAVE_SIZE = CPU_WORKERS
PERSISTENT_WORKERS = False  # 中央MPS方式を切った比較用
THREADS_PER_WORKER = 0  # 0ならCPU数と並列数から自動設定
SCHEDULE_ORDER = 'largest-first'
WARMUP = True
KEEP_BENCHMARK_MODELS = False

RUN_ID = (
    f"{time.strftime('%Y%m%d_%H%M%S')}_"
    f"{time.time_ns() % 1_000_000_000:09d}"
)
OUTPUT_ROOT = MATCH_ROOT / 'results' / 'training_parallel_benchmark' / RUN_ID
display_jobs = 32 if JOB_LIMIT == 0 else JOB_LIMIT
print(
    f'workers={TRAINING_WORKERS}, repeats={REPEATS}, jobs={display_jobs}, '
    f'epochs={EPOCHS}, minibatch={TRAIN_MINIBATCH_SIZE}, device={DEVICE}, '
    f'central_device={CENTRAL_DEVICE_OWNER}, wave={DEVICE_WAVE_SIZE}, '
    f'schedule={SCHEDULE_ORDER}, threads/worker={THREADS_PER_WORKER}, '
    f'output={OUTPUT_ROOT}'
)

workers=[9], repeats=1, jobs=32, epochs=1, minibatch=128, device=auto, central_device=True, wave=9, schedule=largest-first, threads/worker=0, output=/Users/naoki/Desktop/develop/pokemon_tgc_agent/pokemon-tcg-agent/results/training_parallel_benchmark/20260802_002942_873405000


In [12]:
def average_or_nan(values: list[float]) -> float:
    return sum(values) / len(values) if values else float('nan')


MPS_GPU_PATTERN = re.compile(r'"Device Utilization %"=([\d.]+)')


def sample_mps_gpu_percent() -> float | None:
    if sys.platform != 'darwin' or DEVICE not in ('auto', 'mps'):
        return None
    try:
        completed = subprocess.run(
            [
                'ioreg', '-r', '-c', 'AGXAccelerator',
                '-d', '1', '-k', 'PerformanceStatistics',
            ],
            capture_output=True,
            text=True,
            timeout=2,
            check=False,
        )
    except (FileNotFoundError, OSError, subprocess.TimeoutExpired):
        return None
    match = MPS_GPU_PATTERN.search(completed.stdout)
    return float(match.group(1)) if match is not None else None


def process_tree_rss_gib(process_id: int) -> float | None:
    try:
        root = psutil.Process(process_id)
        processes = [root, *root.children(recursive=True)]
        rss = 0
        for process in processes:
            try:
                rss += process.memory_info().rss
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                pass
        return rss / 1024**3
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        return None


def run_training_benchmark(
    workers: int,
    repeat: int,
    *,
    job_limit: int | None = None,
) -> dict[str, float | int | str]:
    measured_job_limit = JOB_LIMIT if job_limit is None else job_limit
    run_label = 'warmup' if repeat < 0 else f'r{repeat + 1}_w{workers}'
    output_dir = OUTPUT_ROOT / run_label
    command = [
        str(PYTHON),
        str(BENCHMARK_RUNNER),
        '--python', str(PYTHON),
        '--train-script', str(TRAIN_SCRIPT),
        '--shards-root', str(SHARDS_ROOT),
        '--agents-root', str(AGENTS_ROOT),
        '--output-dir', str(output_dir),
        '--workers', str(workers),
        '--job-limit', str(measured_job_limit),
        '--epochs', str(EPOCHS),
        '--batch-size', str(TRAIN_MINIBATCH_SIZE),
        '--lr', str(LEARNING_RATE),
        '--device', DEVICE,
        '--seed', str(SEED),
        '--threads-per-worker', str(THREADS_PER_WORKER),
        '--schedule-order', SCHEDULE_ORDER,
        '--device-wave-size', str(DEVICE_WAVE_SIZE),
    ]
    if CENTRAL_DEVICE_OWNER:
        command.append('--central-device-owner')
    if not PERSISTENT_WORKERS:
        command.append('--no-persistent-workers')
    if KEEP_BENCHMARK_MODELS:
        command.append('--keep-models')

    cpu_samples: list[float] = []
    ram_samples: list[float] = []
    available_ram_samples: list[float] = []
    rss_samples: list[float] = []
    gpu_samples: list[float] = []
    psutil.cpu_percent(interval=None)
    started = time.perf_counter()
    process = subprocess.Popen(
        command,
        cwd=MATCH_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding='utf-8',
        errors='replace',
    )
    while process.poll() is None:
        cpu_samples.append(psutil.cpu_percent(interval=0.5))
        memory = psutil.virtual_memory()
        ram_samples.append(memory.percent)
        available_ram_samples.append(memory.available / 1024**3)
        rss = process_tree_rss_gib(process.pid)
        if rss is not None:
            rss_samples.append(rss)
        gpu_percent = sample_mps_gpu_percent()
        if gpu_percent is not None:
            gpu_samples.append(gpu_percent)
    stdout, stderr = process.communicate()
    wall = time.perf_counter() - started
    if process.returncode != 0:
        raise RuntimeError(
            f'{run_label}がexit={process.returncode}で失敗しました。\n'
            f'{stdout[-2000:]}\n{stderr[-2000:]}'
        )

    summary_path = output_dir / 'summary.json'
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    (output_dir / 'launcher.stdout.log').write_text(stdout, encoding='utf-8')
    (output_dir / 'launcher.stderr.log').write_text(stderr, encoding='utf-8')
    job_seconds = [
        float(result['elapsedSeconds']) for result in summary['jobResults']
    ]
    return {
        'training workers': workers,
        'repeat': repeat + 1,
        'jobs': int(summary['jobs']),
        'epochs': int(summary['epochs']),
        '元ゲーム数': int(summary['sourceGames']),
        '学習samples': int(summary['trainedSamples']),
        '実行方式': str(summary['executionMode']),
        'CPU threads/worker': int(summary['threadsPerWorker']),
        '投入順': str(summary['scheduleOrder']),
        'MPS wave': summary['deviceWaveSize'],
        'wall': wall,
        'jobs/s': int(summary['jobs']) / wall,
        'samples/s': int(summary['trainedSamples']) / wall,
        'games/s': int(summary['sourceGames']) / wall,
        '平均job秒': average_or_nan(job_seconds),
        '最大job秒': max(job_seconds, default=float('nan')),
        'CPU平均%': average_or_nan(cpu_samples),
        'CPU最大%': max(cpu_samples, default=float('nan')),
        'GPU平均%': average_or_nan(gpu_samples),
        'GPU最大%': max(gpu_samples, default=float('nan')),
        'RAM最大%': max(ram_samples, default=float('nan')),
        '空きRAM最小GiB': min(available_ram_samples, default=float('nan')),
        'process RSS最大GiB': max(rss_samples, default=float('nan')),
        '出力': str(output_dir),
    }

## 実行

各構成は同じMPSを使うため、構成同士は同時実行せず順番に測定します。`WARMUP=True`では最初に1モデルだけ学習し、MPS初期化時間を本計測から外します。

In [13]:
blocking_processes = []
for candidate in psutil.process_iter(['pid', 'cmdline']):
    if candidate.info['pid'] == os.getpid():
        continue
    command_text = ' '.join(candidate.info.get('cmdline') or [])
    if (
        'run_matches_round_robin.py' in command_text
        or 'train_parallel_generation.py' in command_text
    ):
        blocking_processes.append((candidate.info['pid'], command_text))
if blocking_processes:
    raise RuntimeError(
        '別の対戦・学習が動いているため、終了後に測定してください: '
        + ', '.join(str(pid) for pid, _ in blocking_processes)
    )

if OUTPUT_ROOT.exists():
    raise FileExistsError(
        f'出力先が既にあります。設定セルから再実行してください: {OUTPUT_ROOT}'
    )
OUTPUT_ROOT.mkdir(parents=True)

if WARMUP:
    print('[warmup] 1 worker × 1 job')
    warmup = run_training_benchmark(1, -1, job_limit=1)
    print(f"  wall={warmup['wall']:.2f}s")

rows = []
for repeat in range(REPEATS):
    order = TRAINING_WORKERS if repeat % 2 == 0 else list(reversed(TRAINING_WORKERS))
    for workers in order:
        print(f'[{repeat + 1}/{REPEATS}] training workers={workers}')
        row = run_training_benchmark(workers, repeat)
        rows.append(row)
        print(
            f"  wall={row['wall']:.2f}s, jobs/s={row['jobs/s']:.3f}, "
            f"CPU平均={row['CPU平均%']:.1f}%, "
            f"GPU平均={row['GPU平均%']:.1f}%, "
            f"RSS最大={row['process RSS最大GiB']:.2f}GiB"
        )
raw_results = pd.DataFrame(rows)

[warmup] 1 worker × 1 job
  wall=7.41s
[1/1] training workers=9
  wall=172.42s, jobs/s=0.186, CPU平均=22.2%, GPU平均=2.9%, RSS最大=4.42GiB


In [14]:
summary = (
    raw_results.groupby('training workers', as_index=False)
    .agg(
        execution_mode=('実行方式', 'first'),
        threads_per_worker=('CPU threads/worker', 'first'),
        schedule_order=('投入順', 'first'),
        device_wave_size=('MPS wave', 'first'),
        source_games=('元ゲーム数', 'first'),
        trained_samples=('学習samples', 'mean'),
        wall=('wall', 'mean'),
        wall_min=('wall', 'min'),
        jobs_per_second=('jobs/s', 'mean'),
        samples_per_second=('samples/s', 'mean'),
        games_per_second=('games/s', 'mean'),
        mean_job_seconds=('平均job秒', 'mean'),
        max_job_seconds=('最大job秒', 'max'),
        cpu_mean=('CPU平均%', 'mean'),
        cpu_max=('CPU最大%', 'max'),
        gpu_mean=('GPU平均%', 'mean'),
        gpu_max=('GPU最大%', 'max'),
        ram_max=('RAM最大%', 'max'),
        available_ram_min_gib=('空きRAM最小GiB', 'min'),
        process_rss_max_gib=('process RSS最大GiB', 'max'),
    )
)
baseline_workers = int(summary['training workers'].min())
baseline = float(
    summary.loc[summary['training workers'] == baseline_workers, 'wall'].iloc[0]
)
summary['speedup'] = baseline / summary['wall']
summary['parallel efficiency'] = (
    summary['speedup'] * baseline_workers / summary['training workers']
)
raw_results.to_csv(OUTPUT_ROOT / 'raw_results.csv', index=False)
summary.to_csv(OUTPUT_ROOT / 'summary.csv', index=False)
display(
    summary.style
    .format({
        'wall': '{:.3f}',
        'wall_min': '{:.3f}',
        'jobs_per_second': '{:.3f}',
        'samples_per_second': '{:,.1f}',
        'games_per_second': '{:.2f}',
        'mean_job_seconds': '{:.3f}',
        'max_job_seconds': '{:.3f}',
        'cpu_mean': '{:.1f}',
        'cpu_max': '{:.1f}',
        'gpu_mean': '{:.1f}',
        'gpu_max': '{:.1f}',
        'ram_max': '{:.1f}',
        'available_ram_min_gib': '{:.2f}',
        'process_rss_max_gib': '{:.2f}',
        'speedup': '{:.2f}x',
        'parallel efficiency': '{:.1%}',
    })
    .background_gradient(
        subset=[
            'jobs_per_second', 'samples_per_second',
            'games_per_second', 'speedup',
        ],
        cmap='YlGn',
    )
    .background_gradient(
        subset=['wall', 'mean_job_seconds', 'max_job_seconds'],
        cmap='YlOrRd_r',
    )
)
print(f'saved: {OUTPUT_ROOT}')

,training workers,execution_mode,threads_per_worker,schedule_order,device_wave_size,source_games,trained_samples,wall,wall_min,jobs_per_second,samples_per_second,games_per_second,mean_job_seconds,max_job_seconds,cpu_mean,cpu_max,gpu_mean,gpu_max,ram_max,available_ram_min_gib,process_rss_max_gib,speedup,parallel efficiency
0,9,central-device,1,largest-first,9,3600,1162880.000000,172.419,172.419,0.186,"6,744.5",20.88,45.493,69.106,22.2,66.1,2.9,100.0,78.5,5.17,4.42,1.00x,100.0%


saved: /Users/naoki/Desktop/develop/pokemon_tgc_agent/pokemon-tcg-agent/results/training_parallel_benchmark/20260802_002942_873405000
